In [ ]:
!pip -q install gdown

import gdown, pandas as pd
import re

# **Capacity Data**

## **Refinery Production Capacity Source Data**

Capacity Data Sourced from: https://www.eia.gov/petroleum/refinerycapacity/table3.pdf

In [ ]:
# Original full Google Drive URL
file_url_full = "https://drive.google.com/file/d/1JJJspJS2FQ5BHkqDJYlH0t4YsOZ9HjV9/view?usp=drive_link"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', file_url_full)
if match:
    file_id = match.group(1)
else:
    print("Error: Could not extract file ID from the URL. Please check the provided link.")
    file_id = None # Set to None or handle the error as appropriate

if file_id:
    url = f"https://drive.google.com/uc?id={file_id}"
    out = "refineries.csv"

    try:
        gdown.download(url, out, quiet=False)

        df = pd.read_csv(out)
        print("Successfully loaded CSV file:")
        # display(df.head())
    except Exception as e:
        print(f"An error occurred during download or loading: {e}")
        print("Please ensure the Google Drive file permissions are set to 'Anyone with the link'.")


Downloading...
From: https://drive.google.com/uc?id=1JJJspJS2FQ5BHkqDJYlH0t4YsOZ9HjV9
To: /content/refineries.csv
100%|██████████| 18.3k/18.3k [00:00<00:00, 2.82MB/s]

Successfully loaded CSV file:


In [ ]:
refinery_production = df
print("DataFrame 'df' renamed to 'refinery_production'.")

DataFrame 'df' renamed to 'refinery_production'.


In [ ]:
southern_california_refineries = refinery_production[
    (refinery_production['State'] == 'California') &
    (refinery_production['Location'].isin(['El Segundo', 'Wilmington', 'Carson', 'Torrance']))
]
print("Refineries in Southern California with selected columns:")
display(southern_california_refineries[['State', 'Refiner', 'Location', 'Atmospheric_Crude_Oil_Distillation_Capacity_Barrels_per_Calendar_Day_Operating']])

Refineries in Southern California with selected columns:


,State,Refiner,Location,Atmospheric_Crude_Oil_Distillation_Capacity_Barrels_per_Calendar_Day_Operating
10,California,Chevron USA Inc,El Segundo,285000
15,California,Phillips 66 Company,Wilmington,138700
18,California,Tesoro Refining & Marketing Co,Carson,365000
19,California,Torrance Refining Co LLC,Torrance,160000
20,California,Ultramar Inc,Wilmington,85000
22,California,Valero Refining Co California,Wilmington,6300


**Nelson Complexity Index for Capacity**

Use the Nelson Complexity Index for each refinery to map to simple, hydroskimming, complex refineries and assume certain yields for gasoline and diesel

In [ ]:
YIELDS_BY_CONFIG = {
    "Simple": {
        "gasoline_yield_pct": 5.0,   # e.g., 0–10 typical
        "diesel_yield_pct": 18.0,    # e.g., 10–25 typical
    },
    "Hydroskimming": {
        "gasoline_yield_pct": 28.0,  # e.g., 20–35 typical
        "diesel_yield_pct": 22.0,    # e.g., 15–30 typical
    },
    "Complex": {
        "gasoline_yield_pct": 52.0,  # e.g., 45–60 typical
        "diesel_yield_pct": 28.0,    # e.g., 20–35 typical
    }
}

refinery_yield_configs_df = pd.DataFrame.from_dict(YIELDS_BY_CONFIG, orient='index').reset_index()
refinery_yield_configs_df = refinery_yield_configs_df.rename(columns={'index': 'Refinery_Configuration_Type'})

print("Refinery yield configurations DataFrame created:")
display(refinery_yield_configs_df)

Refinery yield configurations DataFrame created:


,Refinery_Configuration_Type,gasoline_yield_pct,diesel_yield_pct
0,Simple,5.0,18.0
1,Hydroskimming,28.0,22.0
2,Complex,52.0,28.0


In [ ]:
southern_california_refineries = southern_california_refineries.copy()

# Create a dictionary for explicit mappings (Refiner, Location) -> Refinery_Configuration_Type
refinery_config_map = {
    ('Chevron USA Inc', 'El Segundo'): 'Complex',
    ('Phillips 66 Company', 'Wilmington'): 'Complex',
    ('Tesoro Refining & Marketing Co', 'Carson'): 'Complex',
    ('Torrance Refining Co LLC', 'Torrance'): 'Complex',
    ('Ultramar Inc', 'Wilmington'): 'Complex',
    ('Valero Refining Co California', 'Wilmington'): 'Hydroskimming'
}

# Create a combined key for mapping
southern_california_refineries['Refinery_Key'] = list(zip(southern_california_refineries['Refiner'], southern_california_refineries['Location']))

# Apply the mapping, with 'Hydroskimming' as a default if a refinery is not found (though all should be covered here)
southern_california_refineries['Refinery_Configuration_Type'] = southern_california_refineries['Refinery_Key'].map(refinery_config_map).fillna('Hydroskimming')

# Drop the temporary key column
southern_california_refineries = southern_california_refineries.drop(columns=['Refinery_Key'])

print("Refinery configurations assigned to Southern California refineries (explicit mapping):")
display(southern_california_refineries[['Refiner', 'Location', 'Refinery_Configuration_Type']])

Refinery configurations assigned to Southern California refineries (explicit mapping):


,Refiner,Location,Refinery_Configuration_Type
10,Chevron USA Inc,El Segundo,Complex
15,Phillips 66 Company,Wilmington,Complex
18,Tesoro Refining & Marketing Co,Carson,Complex
19,Torrance Refining Co LLC,Torrance,Complex
20,Ultramar Inc,Wilmington,Complex
22,Valero Refining Co California,Wilmington,Hydroskimming


In [ ]:
southern_california_refineries = pd.merge(
    southern_california_refineries,
    refinery_yield_configs_df,
    on='Refinery_Configuration_Type',
    how='left'
)

print("Merged Southern California refineries with yield configurations:")

Merged Southern California refineries with yield configurations:


In [ ]:
southern_california_refineries = southern_california_refineries.rename(columns={'Atmospheric_Crude_Oil_Distillation_Capacity_Barrels_per_Calendar_Day_Operating': 'cdu_capacity_per_day'})
southern_california_refineries['gasoline_barrels_per_cal_day'] = southern_california_refineries['cdu_capacity_per_day'] * (southern_california_refineries['gasoline_yield_pct'] / 100)
southern_california_refineries['diesel_barrels_per_cal_day'] = southern_california_refineries['cdu_capacity_per_day'] * (southern_california_refineries['diesel_yield_pct'] / 100)

print("Southern California refineries with calculated gasoline and diesel production:")
display(southern_california_refineries[[
    'Refiner', 'Location', 'Refinery_Configuration_Type',
    'cdu_capacity_per_day',
    'gasoline_barrels_per_cal_day', 'diesel_barrels_per_cal_day'
]])

Southern California refineries with calculated gasoline and diesel production:


,Refiner,Location,Refinery_Configuration_Type,cdu_capacity_per_day,gasoline_barrels_per_cal_day,diesel_barrels_per_cal_day
0,Chevron USA Inc,El Segundo,Complex,285000,148200.0,79800.0
1,Phillips 66 Company,Wilmington,Complex,138700,72124.0,38836.0
2,Tesoro Refining & Marketing Co,Carson,Complex,365000,189800.0,102200.0
3,Torrance Refining Co LLC,Torrance,Complex,160000,83200.0,44800.0
4,Ultramar Inc,Wilmington,Complex,85000,44200.0,23800.0
5,Valero Refining Co California,Wilmington,Hydroskimming,6300,1764.0,1386.0


In [ ]:
days_in_year = 365

southern_california_refineries['gasoline_per_year'] = southern_california_refineries['gasoline_barrels_per_cal_day'] * days_in_year
southern_california_refineries['diesel_per_year'] = southern_california_refineries['diesel_barrels_per_cal_day'] * days_in_year

total_gasoline_per_year = southern_california_refineries['gasoline_per_year'].sum()
total_diesel_per_year = southern_california_refineries['diesel_per_year'].sum()

total_gasoline_per_cal_day = southern_california_refineries['gasoline_barrels_per_cal_day'].sum()
total_diesel_per_cal_day = southern_california_refineries['diesel_barrels_per_cal_day'].sum()

print(f"Total annual gasoline production for Southern California refineries: {total_gasoline_per_year:,.0f} barrels per year")
print(f"Total annual diesel production for Southern California refineries: {total_diesel_per_year:,.0f} barrels per year")
print('')
print(f"Total daily gasoline production for Southern California refineries: {total_gasoline_per_cal_day:,.0f} barrels per day")
print(f"Total daily diesel production for Southern California refineries: {total_diesel_per_cal_day:,.0f} barrels per day")

Total annual gasoline production for Southern California refineries: 196,840,120 barrels per year
Total annual diesel production for Southern California refineries: 106,150,030 barrels per year

Total daily gasoline production for Southern California refineries: 539,288 barrels per day
Total daily diesel production for Southern California refineries: 290,822 barrels per day


## **Refinery Storage Capacity Source Data**

In [ ]:
## hehe need to find this data somewhere

## **Port Inflow Source Data**

*Will need to determine the theoretical capacity (as a function of levers)*

**Port of Los Angeles**

Data Sourced from: https://portoflosangeles.org/business/statistics/tonnage-statistics/historical-tonnage-statistics-2025

In [ ]:
# Original full Google Drive URL
file_url_full = "https://drive.google.com/file/d/169OcOpMgUrh9PRMTU0hHyZNemDC5bZsP/view?usp=drive_link"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', file_url_full)
if match:
    file_id = match.group(1)
else:
    print("Error: Could not extract file ID from the URL. Please check the provided link.")
    file_id = None # Set to None or handle the error as appropriate

if file_id:
    url = f"https://drive.google.com/uc?id={file_id}"
    out = "pola.csv"

    try:
        gdown.download(url, out, quiet=False)

        df = pd.read_csv(out)
        print("Successfully loaded CSV file:")
        # display(df.head())
    except Exception as e:
        print(f"An error occurred during download or loading: {e}")
        print("Please ensure the Google Drive file permissions are set to 'Anyone with the link'.")


Downloading...
From: https://drive.google.com/uc?id=169OcOpMgUrh9PRMTU0hHyZNemDC5bZsP
To: /content/pola.csv
100%|██████████| 821/821 [00:00<00:00, 2.50MB/s]

Successfully loaded CSV file:


In [ ]:
port_la_imports = df
print("DataFrame 'df' renamed to 'port_la_imports'.")
port_la_imports

DataFrame 'df' renamed to 'port_la_imports'.


,Year,Month,Crude_Oil_barrels,Bunkers_barrels,Refined_Products_Barge_barrels,Refined_Products_Pipeline_barrels,Gasoline_barrels,Jet_Fuel_barrels,Monthly_Total_barrels
0,2025,January,0,0,1293052,0,937376,830230,3060658
1,2025,February,375716,0,970437,0,73419,1105502,2525074
2,2025,March,209461,0,1006575,101204,1088292,1180232,3585764
3,2025,April,402754,0,1399738,99718,752023,1141349,3795582
4,2025,May,0,0,1433413,243480,357915,1408873,3443681
5,2025,June,0,0,1587776,178757,508866,785797,3061196
6,2025,July,520617,0,1363103,258193,386658,999510,3528081
7,2025,August,0,0,2052498,87697,457526,1317605,3915326
8,2025,September,0,0,1699701,214183,231802,1545007,3690693
9,2025,October,0,0,1307982,821333,546552,931749,3607616


In [ ]:
days_in_year = 365

total_annual_gasoline_imports = port_la_imports['Gasoline_barrels'].sum()
average_daily_gasoline_imports = total_annual_gasoline_imports / days_in_year

# Assuming 20% of 'Refined_Products_Barge_barrels' is diesel
diesel_percentage = 0.20

total_annual_refined_barge_imports = port_la_imports['Refined_Products_Barge_barrels'].sum() # Added this line
total_annual_refined_barge_diesel = total_annual_refined_barge_imports * diesel_percentage
average_daily_refined_barge_diesel = total_annual_refined_barge_diesel / days_in_year


print(f"Total annual gasoline imports for Port of Los Angeles: {total_annual_gasoline_imports:,.0f} barrels")
print(f"Total annual diesel imports for Port of Los Angeles: {total_annual_refined_barge_diesel:,.0f} barrels")
print('')
print(f"Average daily gasoline imports for Port of Los Angeles: {average_daily_gasoline_imports:,.0f} barrels per day")
print(f"Average daily diesel imports for Port of Los Angeles: {average_daily_refined_barge_diesel:,.0f} barrels per day")

Total annual gasoline imports for Port of Los Angeles: 6,863,513 barrels
Total annual diesel imports for Port of Los Angeles: 3,442,725 barrels

Average daily gasoline imports for Port of Los Angeles: 18,804 barrels per day
Average daily diesel imports for Port of Los Angeles: 9,432 barrels per day


**Port of Long Beach**

Data Sourced from: https://ndc.ops.usace.army.mil/wcsc/webpub/?utm_source=chatgpt.com#/report-landing/year/2023/region/4/location/4110

In [ ]:
# Original full Google Drive URL
file_url_full = "https://drive.google.com/file/d/1qoIrdG8wgnR4DQJTeMbmV4gnMG-X-KY4/view?usp=drive_link"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', file_url_full)
if match:
    file_id = match.group(1)
else:
    print("Error: Could not extract file ID from the URL. Please check the provided link.")
    file_id = None # Set to None or handle the error as appropriate

if file_id:
    url = f"https://drive.google.com/uc?id={file_id}"
    out = "polb.csv"

    try:
        gdown.download(url, out, quiet=False)

        df = pd.read_csv(out)
        print("Successfully loaded CSV file:")
        # display(df.head())
    except Exception as e:
        print(f"An error occurred during download or loading: {e}")
        print("Please ensure the Google Drive file permissions are set to 'Anyone with the link'.")

Downloading...
From: https://drive.google.com/uc?id=1qoIrdG8wgnR4DQJTeMbmV4gnMG-X-KY4
To: /content/polb.csv
100%|██████████| 17.1k/17.1k [00:00<00:00, 31.7MB/s]

Successfully loaded CSV file:


In [ ]:
port_lb_imports = df
print("DataFrame 'df' renamed to 'port_lb_imports'.")
port_lb_imports

DataFrame 'df' renamed to 'port_lb_imports'.


,Code,Commodity,All Traffic Directions CY2023,All Traffic Directions CY2022,All Traffic Directions CY2021,All Traffic Directions CY2020,All Traffic Directions CY2019,Intraport CY2023,Intraport CY2022,Intraport CY2021,...,Receipts CY2023,Receipts CY2022,Receipts CY2021,Receipts CY2020,Receipts CY2019,Shipments CY2023,Shipments CY2022,Shipments CY2021,Shipments CY2020,Shipments CY2019
0,0,All Commodities,85260786,92958926,91501826,79178087,80376848,340758,248481,384064,...,61836270,68224394,67352319,57217641,57246803,23083758,24486051,23765443,21793508,23098711
1,1100,Coal & Lignite,658536,1052941,1026372,449145,1449431,0,0,0,...,518,6365,8691,11511,9498,658018,1046576,1017681,437634,1439933
2,1200,Coal Coke,0,84891,36377,12032,24382,0,0,0,...,0,0,1,15,0,0,84891,36376,12017,24382
3,2100,Crude Petroleum,29165480,28611315,26780341,22260054,25302056,0,0,0,...,28846409,28611315,26780062,22226436,25256641,319071,0,279,33618,31123
4,2211,Gasoline,839873,1312073,1435776,1731071,942115,0,0,0,...,425105,996827,856542,890936,920185,414768,315246,579234,820187,21930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,7600,Rubber & Plastic Pr.,2948426,3492379,2993241,2299960,2125675,0,0,0,...,2782681,3348399,2853443,2125920,1879616,165745,143980,139798,174040,246059
138,7800,Empty Containers,0,5,0,0,0,0,0,0,...,0,5,0,0,0,0,0,0,0,0
139,7900,Manufac. Prod. NEC,8314631,10057721,10447124,8299383,6696538,0,0,0,...,6378234,7796268,7994043,6496914,5615990,1936397,2261453,2453081,1802469,1080548
140,8900,Waste and Scrap NEC,20,0,18,103,661,0,0,0,...,12,0,18,97,125,8,0,0,6,536


In [ ]:
port_lb_imports

,Code,Commodity,All Traffic Directions CY2023,All Traffic Directions CY2022,All Traffic Directions CY2021,All Traffic Directions CY2020,All Traffic Directions CY2019,Intraport CY2023,Intraport CY2022,Intraport CY2021,...,Receipts CY2023,Receipts CY2022,Receipts CY2021,Receipts CY2020,Receipts CY2019,Shipments CY2023,Shipments CY2022,Shipments CY2021,Shipments CY2020,Shipments CY2019
0,0,All Commodities,85260786,92958926,91501826,79178087,80376848,340758,248481,384064,...,61836270,68224394,67352319,57217641,57246803,23083758,24486051,23765443,21793508,23098711
1,1100,Coal & Lignite,658536,1052941,1026372,449145,1449431,0,0,0,...,518,6365,8691,11511,9498,658018,1046576,1017681,437634,1439933
2,1200,Coal Coke,0,84891,36377,12032,24382,0,0,0,...,0,0,1,15,0,0,84891,36376,12017,24382
3,2100,Crude Petroleum,29165480,28611315,26780341,22260054,25302056,0,0,0,...,28846409,28611315,26780062,22226436,25256641,319071,0,279,33618,31123
4,2211,Gasoline,839873,1312073,1435776,1731071,942115,0,0,0,...,425105,996827,856542,890936,920185,414768,315246,579234,820187,21930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,7600,Rubber & Plastic Pr.,2948426,3492379,2993241,2299960,2125675,0,0,0,...,2782681,3348399,2853443,2125920,1879616,165745,143980,139798,174040,246059
138,7800,Empty Containers,0,5,0,0,0,0,0,0,...,0,5,0,0,0,0,0,0,0,0
139,7900,Manufac. Prod. NEC,8314631,10057721,10447124,8299383,6696538,0,0,0,...,6378234,7796268,7994043,6496914,5615990,1936397,2261453,2453081,1802469,1080548
140,8900,Waste and Scrap NEC,20,0,18,103,661,0,0,0,...,12,0,18,97,125,8,0,0,6,536


In [ ]:
days_in_year = 365

gasoline_receipts_2023 = port_lb_imports[port_lb_imports['Commodity'] == 'Gasoline']['Receipts CY2023'].iloc[0]
distillate_fuel_oil_receipts_2023 = port_lb_imports[port_lb_imports['Commodity'] == 'Distillate Fuel Oil']['Receipts CY2023'].iloc[0]

average_daily_gasoline_receipts_2023 = gasoline_receipts_2023 / days_in_year
average_daily_distillate_fuel_oil_receipts_2023 = distillate_fuel_oil_receipts_2023 / days_in_year

print(f"Receipts for Gasoline in 2023 (Port of Long Beach): {gasoline_receipts_2023:,.0f}")
print(f"Receipts for Distillate Fuel Oil in 2023 (Port of Long Beach): {distillate_fuel_oil_receipts_2023:,.0f}")
print('')
print(f"Average daily receipts for Gasoline in 2023 (Port of Long Beach): {average_daily_gasoline_receipts_2023:,.0f}")
print(f"Average daily receipts for Distillate Fuel Oil in 2023 (Port of Long Beach): {average_daily_distillate_fuel_oil_receipts_2023:,.0f}")

Receipts for Gasoline in 2023 (Port of Long Beach): 425,105
Receipts for Distillate Fuel Oil in 2023 (Port of Long Beach): 3,123,439

Average daily receipts for Gasoline in 2023 (Port of Long Beach): 1,165
Average daily receipts for Distillate Fuel Oil in 2023 (Port of Long Beach): 8,557


**Combined Port Imports**


In [ ]:
# Gather annual gasoline and diesel data for each port
port_data = {
    'Port': ['Port of Los Angeles', 'Port of Long Beach'],
    'Gasoline_Annual_Barrels': [total_annual_gasoline_imports, gasoline_receipts_2023],
    'Diesel_Annual_Barrels': [total_annual_refined_barge_diesel, distillate_fuel_oil_receipts_2023]
}

combined_port_imports_df = pd.DataFrame(port_data)

print("Combined Annual Imports for Ports:")
display(combined_port_imports_df)

Combined Annual Imports for Ports:


,Port,Gasoline_Annual_Barrels,Diesel_Annual_Barrels
0,Port of Los Angeles,6863513,3442725.2
1,Port of Long Beach,425105,3123439.0


In [ ]:
# Gather annual gasoline and diesel data for each port
port_data = {
    'Port': ['Port of Los Angeles', 'Port of Long Beach'],
    'Gasoline_Annual_Barrels': [total_annual_gasoline_imports, gasoline_receipts_2023],
    'Diesel_Annual_Barrels': [total_annual_refined_barge_diesel, distillate_fuel_oil_receipts_2023]
}

combined_port_imports_df = pd.DataFrame(port_data)
# Convert relevant columns to integer for no decimals
combined_port_imports_df['Gasoline_Annual_Barrels'] = combined_port_imports_df['Gasoline_Annual_Barrels'].astype(int)
combined_port_imports_df['Diesel_Annual_Barrels'] = combined_port_imports_df['Diesel_Annual_Barrels'].astype(int)

print("Combined Annual Imports for Ports:")
display(combined_port_imports_df)

# Gather daily gasoline and diesel data for each port
daily_port_data = {
    'Port': ['Port of Los Angeles', 'Port of Long Beach'],
    'Gasoline_Daily_Barrels': [average_daily_gasoline_imports, average_daily_gasoline_receipts_2023],
    'Diesel_Daily_Barrels': [average_daily_refined_barge_diesel, average_daily_distillate_fuel_oil_receipts_2023]
}

combined_daily_port_imports_df = pd.DataFrame(daily_port_data)
# Convert relevant columns to integer for no decimals
combined_daily_port_imports_df['Gasoline_Daily_Barrels'] = combined_daily_port_imports_df['Gasoline_Daily_Barrels'].astype(int)
combined_daily_port_imports_df['Diesel_Daily_Barrels'] = combined_daily_port_imports_df['Diesel_Daily_Barrels'].astype(int)

print("\nCombined Daily Imports for Ports:")
display(combined_daily_port_imports_df)

Combined Annual Imports for Ports:


,Port,Gasoline_Annual_Barrels,Diesel_Annual_Barrels
0,Port of Los Angeles,6863513,3442725
1,Port of Long Beach,425105,3123439



Combined Daily Imports for Ports:


,Port,Gasoline_Daily_Barrels,Diesel_Daily_Barrels
0,Port of Los Angeles,18804,9432
1,Port of Long Beach,1164,8557


## **Port Storage Capacity Source Data**

In [ ]:
## hehe also need to find this data

## **Terminals Operational Flow Capacity Data**

In [ ]:
# Original full Google Drive URL
file_url_full = "https://drive.google.com/file/d/1DHiXbAvXsJV0MPAwCKIQoqTn1lYFVayS/view?usp=drive_link"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', file_url_full)
if match:
    file_id = match.group(1)
else:
    print("Error: Could not extract file ID from the URL. Please check the provided link.")
    file_id = None # Set to None or handle the error as appropriate

if file_id:
    url = f"https://drive.google.com/uc?id={file_id}"
    out = "terminals.csv"

    try:
        gdown.download(url, out, quiet=False)

        df = pd.read_csv(out)
        print("Successfully loaded CSV file:")
        # display(df.head())
    except Exception as e:
        print(f"An error occurred during download or loading: {e}")
        print("Please ensure the Google Drive file permissions are set to 'Anyone with the link'.")


Downloading...
From: https://drive.google.com/uc?id=1DHiXbAvXsJV0MPAwCKIQoqTn1lYFVayS
To: /content/terminals.csv
100%|██████████| 1.71k/1.71k [00:00<00:00, 5.37MB/s]

Successfully loaded CSV file:


In [ ]:
terminals_ofc = df
print("DataFrame 'df' renamed to 'terminals_ofc'.")
terminals_ofc

DataFrame 'df' renamed to 'terminals_ofc'.


,Terminal group,Terminal name,Company,Bays,Gates,Delivery Radius,Radius rounded,Key Gates-Bays,OFC (MMGal/Day),Primary group name,Leadtime
0,858,Bakersfield,Bakersfield Terminal Group,8,1,50,50,1-8,1.5,Bakersfield,4
1,858,Bakersfield,Tricor Refining,15,1,50,50,1-15,2.0,Bakersfield,4
2,858,Shafter,Plains All American Pipeline,8,1,50,50,1-8,1.5,Bakersfield,4
3,860,Barstow,Kinder Morgan,4,2,72,75,2-4,1.0,Barstow,4
4,870,Colton,Arco/Maraathon,5,1,31,25,1-5,1.0,Colton,2
5,870,Colton A,Kinder Morgan,4,1,31,25,1-4,1.0,Colton,2
6,870,Colton B,Kinder Morgan,5,1,31,25,1-5,1.0,Colton,2
7,870,Colton C,Kinder Morgan,4,1,31,25,1-4,1.0,Colton,2
8,870,Colton,Phillips 66,5,2,31,25,2-5,2.0,Colton,2
9,900,Imperial,Kinder Morgan,9,2,48,50,2-9,2.0,Imperial,4


In [ ]:
gallons_per_barrel = 42
million_to_unit_factor = 1_000_000

terminals_ofc['OFC_barrels_per_day'] = (terminals_ofc['OFC (MMGal/Day)'] * million_to_unit_factor) / gallons_per_barrel

print("Terminals Operational Flow Capacity with Barrels per Day:")
display(terminals_ofc[['Primary group name','Terminal name', 'Company', 'OFC (MMGal/Day)', 'OFC_barrels_per_day']])

Terminals Operational Flow Capacity with Barrels per Day:


,Primary group name,Terminal name,Company,OFC (MMGal/Day),OFC_barrels_per_day
0,Bakersfield,Bakersfield,Bakersfield Terminal Group,1.5,35714.285714
1,Bakersfield,Bakersfield,Tricor Refining,2.0,47619.047619
2,Bakersfield,Shafter,Plains All American Pipeline,1.5,35714.285714
3,Barstow,Barstow,Kinder Morgan,1.0,23809.523810
4,Colton,Colton,Arco/Maraathon,1.0,23809.523810
5,Colton,Colton A,Kinder Morgan,1.0,23809.523810
6,Colton,Colton B,Kinder Morgan,1.0,23809.523810
7,Colton,Colton C,Kinder Morgan,1.0,23809.523810
8,Colton,Colton,Phillips 66,2.0,47619.047619
9,Imperial,Imperial,Kinder Morgan,2.0,47619.047619


In [ ]:
terminal_group_summary = terminals_ofc.groupby('Primary group name')['OFC_barrels_per_day'].sum().reset_index()
terminal_group_summary['OFC_barrels_per_day'] = terminal_group_summary['OFC_barrels_per_day'].astype(int)
terminal_group_summary = terminal_group_summary.rename(columns={'OFC_barrels_per_day': 'Total OFC (barrels/day)'})
print("Summary of Primary Group Names and their total OFC (barrels/day):")
display(terminal_group_summary)

Summary of Primary Group Names and their total OFC (barrels/day):


,Primary group name,Total OFC (barrels/day)
0,Bakersfield,119047
1,Barstow,23809
2,Colton,142857
3,Imperial,47619
4,Los Angeles,285714
5,Orange,166666
6,San Diego,95238


## **Summary**

In [ ]:
# Calculate Total Refinery Supply
total_refinery_gasoline = southern_california_refineries['gasoline_barrels_per_cal_day'].sum()
total_refinery_diesel = southern_california_refineries['diesel_barrels_per_cal_day'].sum()
total_refinery_supply = total_refinery_gasoline + total_refinery_diesel

# Calculate Total Port Imports
total_port_gasoline_imports = combined_daily_port_imports_df['Gasoline_Daily_Barrels'].sum()
total_port_diesel_imports = combined_daily_port_imports_df['Diesel_Daily_Barrels'].sum()
total_port_supply = total_port_gasoline_imports + total_port_diesel_imports

# Calculate Grand Total Supply
grand_total_supply = total_refinery_supply + total_port_supply

# Calculate Total Terminal Demand (OFC)
total_terminal_ofc = terminal_group_summary['Total OFC (barrels/day)'].sum()

# Calculate Balance
balance = grand_total_supply - total_terminal_ofc

print(f"\nSupply (barrels per day):\n")
print(f"  Refinery Production:")
print(f"    Gasoline: {total_refinery_gasoline:,.0f} barrels/day")
print(f"    Diesel: {total_refinery_diesel:,.0f} barrels/day")
print(f"    Total Refinery Supply: {total_refinery_supply:,.0f} barrels/day")
print(f"\n  Port Imports:")
print(f"    Gasoline: {total_port_gasoline_imports:,.0f} barrels/day")
print(f"    Diesel: {total_port_diesel_imports:,.0f} barrels/day")
print(f"    Total Port Imports: {total_port_supply:,.0f} barrels/day")
print(f"\n  Grand Total Supply (Refineries + Ports): {grand_total_supply:,.0f} barrels/day")

print(f"\nDemand (barrels per day):\n")
print(f"  Terminal Operational Flow Capacity (OFC): {total_terminal_ofc:,.0f} barrels/day")

print(f"\nComparison:\n")
print(f"The total estimated daily supply of gasoline and diesel (including refinery production and port imports) is approximately {grand_total_supply:,.0f} barrels per day. The total estimated daily demand from terminal operational flow capacity is approximately {total_terminal_ofc:,.0f} barrels per day. The difference is {balance:,.0f} barrels per day.")
print("\n*Note: The terminal operational flow capacity represents the total capacity for petroleum products and is not disaggregated by gasoline and diesel. Thus, this balance provides a high-level overview.*")


Supply (barrels per day):

  Refinery Production:
    Gasoline: 539,288 barrels/day
    Diesel: 290,822 barrels/day
    Total Refinery Supply: 830,110 barrels/day

  Port Imports:
    Gasoline: 19,968 barrels/day
    Diesel: 17,989 barrels/day
    Total Port Imports: 37,957 barrels/day

  Grand Total Supply (Refineries + Ports): 868,067 barrels/day

Demand (barrels per day):

  Terminal Operational Flow Capacity (OFC): 880,950 barrels/day

Comparison:

The total estimated daily supply of gasoline and diesel (including refinery production and port imports) is approximately 868,067 barrels per day. The total estimated daily demand from terminal operational flow capacity is approximately 880,950 barrels per day. The difference is -12,883 barrels per day.

*Note: The terminal operational flow capacity represents the total capacity for petroleum products and is not disaggregated by gasoline and diesel. Thus, this balance provides a high-level overview.*


# **Mapping Data**

## **Pipelines**

In [ ]:
# Original full Google Drive URL
file_url_full = "https://drive.google.com/file/d/17e3vpT3-ZMtEUCf2CoLBvtkAO3iGZO8a/view?usp=drive_link"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', file_url_full)
if match:
    file_id = match.group(1)
else:
    print("Error: Could not extract file ID from the URL. Please check the provided link.")
    file_id = None # Set to None or handle the error as appropriate

if file_id:
    url = f"https://drive.google.com/uc?id={file_id}"
    out = "pipeline_hierarchy.csv"

    try:
        gdown.download(url, out, quiet=False)

        df = pd.read_csv(out)
        print("Successfully loaded CSV file:")
        # display(df.head())
    except Exception as e:
        print(f"An error occurred during download or loading: {e}")
        print("Please ensure the Google Drive file permissions are set to 'Anyone with the link'.")


Downloading...
From: https://drive.google.com/uc?id=17e3vpT3-ZMtEUCf2CoLBvtkAO3iGZO8a
To: /content/pipeline_hierarchy.csv
100%|██████████| 262/262 [00:00<00:00, 437kB/s]

Successfully loaded CSV file:


In [ ]:
pipeline_hierarchy = df
print("DataFrame 'df' renamed to 'pipeline_hierarchy'.")
pipeline_hierarchy

DataFrame 'df' renamed to 'pipeline_hierarchy'.


,id,name,daily_capacity,lead_time_days,upstream
0,A_shared,A_shared,1000,1,NaN
1,A_main,A_main,1000,3,A_shared
2,A_delay_T2,A_delay_T2,10000000000,3,A_main
3,C_main,C_main,1000,5,A_shared
4,B_main,B_main,1000,4,NaN
5,C_delay_T5,C_delay_T5,10000000000,3,C_main
6,D_main,D_main,300,1,NaN


## **Terminals from Pipeline**

In [ ]:
# Original full Google Drive URL
file_url_full = "https://drive.google.com/file/d/1Q6EbB8Za3cw0dEKMnbJq5KW45zFLexdS/view?usp=drive_link"

# Extract the file ID using a regular expression
match = re.search(r'/d/([a-zA-Z0-9_-]+)', file_url_full)
if match:
    file_id = match.group(1)
else:
    print("Error: Could not extract file ID from the URL. Please check the provided link.")
    file_id = None # Set to None or handle the error as appropriate

if file_id:
    url = f"https://drive.google.com/uc?id={file_id}"
    out = "refinery_to_pipeline.csv"

    try:
        gdown.download(url, out, quiet=False)

        df = pd.read_csv(out)
        print("Successfully loaded CSV file:")
        # display(df.head())
    except Exception as e:
        print(f"An error occurred during download or loading: {e}")
        print("Please ensure the Google Drive file permissions are set to 'Anyone with the link'.")


An error occurred during download or loading: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1Q6EbB8Za3cw0dEKMnbJq5KW45zFLexdS

but Gdown can't. Please check connections and permissions.
Please ensure the Google Drive file permissions are set to 'Anyone with the link'.


In [ ]:
terminals_from_pipeline_hierarchy = df
print("DataFrame 'df' renamed to 'terminals_from_pipeline_hierarchy'.")
terminals_from_pipeline_hierarchy

DataFrame 'df' renamed to 'terminals_from_pipeline_hierarchy'.


,id,name,daily_capacity,lead_time_days,upstream
0,A_shared,A_shared,1000,1,NaN
1,A_main,A_main,1000,3,A_shared
2,A_delay_T2,A_delay_T2,10000000000,3,A_main
3,C_main,C_main,1000,5,A_shared
4,B_main,B_main,1000,4,NaN
5,C_delay_T5,C_delay_T5,10000000000,3,C_main
6,D_main,D_main,300,1,NaN


## Connected Network


Generate a directed network diagram using `NetworkX` and `Matplotlib` to visualize the refinery-to-pipeline-to-terminal connections. Install `Graphviz` and `Pygraphviz`, then create a graph that includes a 'Refineries' node, pipeline nodes from `pipeline_hierarchy`, and terminal nodes from `terminals_from_pipeline_hierarchy`. Add edges connecting 'Refineries' to initial pipelines (`A_shared`, `B_main`, `D_main`), pipelines to their downstream counterparts, and pipelines to their respective terminals, labeling pipeline edges with their IDs. Force a left-to-right layout using `nx.nx_agraph.graphviz_layout(prog='dot', args='-Grankdir=LR')`. Style the 'Refineries' node as a square, pipeline nodes with a distinct shape and color, and terminal nodes as circles with another color. Add node labels, edge labels, a title, and a legend.

In [ ]:
!sudo apt-get install graphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import sys
!{sys.executable} -m pip install pygraphviz --no-binary :all:


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pygraphviz (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pygraphviz
Failed to build pygraphviz
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pygraphviz)


In [ ]:
!sudo apt-get install -y libgraphviz-dev
import sys
!{sys.executable} -m pip install pygraphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgtk2.0-0
  libgtk2.0-bin libgtk2.0-common libgvc6-plugins-gtk librsvg2-common
  libxcomposite1 libxdot4
Suggested packages:
  gvfs
The following NEW packages will be installed:
  libatk1.0-0 libatk1.0-data libgail-common libgail18 libgraphviz-dev
  libgtk2.0-0 libgtk2.0-bin libgtk2.0-common libgvc6-plugins-gtk
  librsvg2-common libxcomposite1 libxdot4
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 2,496 kB of archives.
After this operation, 7,963 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libgtk2.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print("NetworkX and Matplotlib imported successfully.")

NetworkX and Matplotlib imported successfully.


In [ ]:
G = nx.DiGraph()

# 1. Add the central 'Refineries' node
G.add_node('Refineries', node_type='source')

# 2. Add pipeline nodes and edges from pipeline_hierarchy
for index, row in pipeline_hierarchy.iterrows():
    pipeline_id = row['id']
    pipeline_name = row['name']
    upstream_id = row['upstream']

    G.add_node(pipeline_id, node_type='pipeline', label=pipeline_name)

    if pd.isna(upstream_id): # If no upstream, connect to 'Refineries'
        # Check if 'Refineries' already has an edge to this pipeline_id
        # This handles cases like 'A_shared', 'B_main', 'D_main' which are initial pipelines
        if not G.has_edge('Refineries', pipeline_id):
            G.add_edge('Refineries', pipeline_id, label=pipeline_id)
    else:
        G.add_edge(upstream_id, pipeline_id, label=pipeline_id)

# 3. Add terminal nodes and edges from terminals_from_pipeline_hierarchy
for index, row in terminals_from_pipeline_hierarchy.iterrows():
    terminal_name = row['terminal']
    pipeline_id = row['pipeline_id']

    G.add_node(terminal_name, node_type='terminal', label=terminal_name)
    G.add_edge(pipeline_id, terminal_name, label=pipeline_id)


print("Graph initialized and nodes/edges added.")

KeyError: 'terminal'

In [ ]:
plt.figure(figsize=(18, 10))

# Define layout for left-to-right direction
# Use 'dot' engine for Graphviz and '-Grankdir=LR' for left-to-right layout
pos = nx.nx_agraph.graphviz_layout(G, prog='dot', args='-Grankdir=LR')

# Define node colors and shapes based on node_type
node_colors = []
node_shapes = {}
for node, data in G.nodes(data=True):
    if data['node_type'] == 'source':
        node_colors.append('lightcoral') # Color for 'Refineries'
        node_shapes[node] = 's' # Square for 'Refineries'
    elif data['node_type'] == 'pipeline':
        node_colors.append('lightgreen') # Color for pipelines
        node_shapes[node] = 'o' # Circle for pipelines
    elif data['node_type'] == 'terminal':
        node_colors.append('skyblue') # Color for terminals
        node_shapes[node] = 'o' # Circle for terminals

# Prepare node labels (using the 'label' attribute if available)
labels = {node: G.nodes[node].get('label', node) for node in G.nodes()}

# Draw nodes
# We need to draw nodes by shape since nx.draw_networkx_nodes only accepts one shape at a time
for shape_type in set(node_shapes.values()):
    nodes_of_shape = [node for node, shape in node_shapes.items() if shape == shape_type]
    colors_of_shape = [node_colors[list(G.nodes()).index(node)] for node in nodes_of_shape]
    nx.draw_networkx_nodes(G, pos, nodelist=nodes_of_shape, node_shape=shape_type, node_color=colors_of_shape, node_size=3000)

# Draw edges with arrows
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, edge_color='gray')

# Draw node labels
nx.draw_networkx_labels(G, pos, labels, font_size=10)

# Draw edge labels
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='blue', font_size=8)

# Add a title
plt.title('Refinery to Pipeline to Terminal Network Diagram', size=15)

# Create a legend
import matplotlib.patches as mpatches
red_patch = mpatches.Patch(color='lightcoral', label='Refineries (Source)')
green_patch = mpatches.Patch(color='lightgreen', label='Pipelines')
blue_patch = mpatches.Patch(color='skyblue', label='Terminals')
plt.legend(handles=[red_patch, green_patch, blue_patch], loc='lower right', fontsize=12)

plt.box(False)
plt.show()

print("Network diagram generated successfully.")

# Task
Identify the liquid petroleum product commodities in the 'Commodity' column of the `port_lb_imports` DataFrame and list them.

## Identify liquid commodities

### Subtask:
Examine the 'Commodity' column in the `port_lb_imports` DataFrame and identify entries that represent liquid petroleum products, similar to gasoline.


**Reasoning**:
The first instruction is to extract and display the unique values from the 'Commodity' column of the `port_lb_imports` DataFrame to understand the types of commodities present.



In [ ]:
unique_commodities = port_lb_imports['Commodity'].unique()
print("Unique Commodities in Port of Long Beach Data:\n")
for commodity in unique_commodities:
    print(commodity)

After reviewing the unique commodities from the `port_lb_imports` DataFrame, the following entries are identified as liquid petroleum products:

*   `Crude Petroleum`
*   `Gasoline`
*   `Kerosene`
*   `Distillate Fuel Oil`
*   `Residual Fuel Oil`
*   `Lube Oil & Greases`
*   `Naphtha & Solvents`
*   `Asphalt; Tar & Pitch`
*   `Petro. Jelly & Waxes`
*   `Hydrocarbon & Petrol Gases; Liquefied and Gaseous` (though some are gaseous, they are often transported in liquid form or as part of petroleum products)
*   `Petro. Products NEC` (Petroleum Products Not Elsewhere Classified)

## Summary:

### Q&A
The identified liquid petroleum product commodities in the 'Commodity' column of the `port_lb_imports` DataFrame are:
*   `Crude Petroleum`
*   `Gasoline`
*   `Kerosene`
*   `Distillate Fuel Oil`
*   `Residual Fuel Oil`
*   `Lube Oil & Greases`
*   `Naphtha & Solvents`
*   `Asphalt; Tar & Pitch`
*   `Petro. Jelly & Waxes`
*   `Hydrocarbon & Petrol Gases; Liquefied and Gaseous`
*   `Petro. Products NEC` (Petroleum Products Not Elsewhere Classified)

### Data Analysis Key Findings
*   A total of 11 unique commodities were identified as liquid petroleum products from the `port_lb_imports` DataFrame.
*   These identified commodities include well-known petroleum products such as `Crude Petroleum`, `Gasoline`, and `Kerosene`, as well as broader categories like `Petro. Products NEC`.
*   The list also includes products that can be liquid or liquefied, such as `Hydrocarbon & Petrol Gases; Liquefied and Gaseous`.

### Insights or Next Steps
*   This identified list of liquid commodities can be used to filter the `port_lb_imports` DataFrame for further analysis specific to these products, such as import volume trends or comparisons.
*   Consider refining the classification of `Hydrocarbon & Petrol Gases` to explicitly distinguish between liquid and gaseous forms if the dataset allows for more precise analysis of liquid-only transport.
